# Project Requirement Feasibility Check

Quick, cheap sanity check against the [Project Requirements](../../README.md#project-requirements) in the README, using the raw/interim files we already have for all three sources — **before** building full profiling/extraction/EDA pipelines for OurAirports and Kaggle the way we did for Wikidata. If the numbers here look comfortable, that's justification to go build those pipelines properly; if something's borderline, better to know now.

**Sources used**:
- `data/raw/ourairports/airports-ourairports.csv` (raw download, untouched)
- `data/raw/kaggle/airports-kaggle.csv` (raw download, untouched)
- `data/interim/wikidata_airports.csv` (already resolved/fused — the output of `wikidata_extraction.ipynb`)

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

oa = pd.read_csv(project_root / "data" / "raw" / "ourairports" / "airports-ourairports.csv")
kg = pd.read_csv(project_root / "data" / "raw" / "kaggle" / "airports-kaggle.csv")
wd = pd.read_csv(project_root / "data" / "interim" / "wikidata_airports.csv")

print(f"OurAirports: {len(oa):>7,} rows, {oa.shape[1]} columns")
print(f"Kaggle:      {len(kg):>7,} rows, {kg.shape[1]} columns")
print(f"Wikidata:    {len(wd):>7,} rows, {wd.shape[1]} columns")

OurAirports:  86,151 rows, 19 columns
Kaggle:        6,393 rows, 13 columns
Wikidata:     24,871 rows, 13 columns


## Requirement 1 — 3 different data sets

Satisfied: OurAirports (CSV bulk download), Kaggle (CSV dataset), Wikidata (SPARQL via QLever).

## Attribute mapping across sources

Manual mapping from source-specific column names to a shared conceptual attribute, for requirements 4 and 5.

| Attribute | OurAirports | Kaggle | Wikidata | # sources |
|---|---|---|---|---|
| name | `name` | `AirportName` | `name` | 3 |
| iata | `iata_code` | `IATA` | `iata` | 3 |
| icao | `icao_code` | `ICAO` | `icao` | 3 |
| coordinates | `latitude_deg`/`longitude_deg` | `GeoPointLat`/`GeoPointLong` | `lat`/`lon` | 3 |
| country | `iso_country` | `Country_Name`/`Country_CodeA2` | `country` | 3 |
| city/municipality | `municipality` | `City_Name` | `located_in` | 3 |
| elevation | `elevation_ft` | — | `elevation` (m) | 2 |
| timezone | — | `TimeZone` | `timezone` | 2 |
| local/FAA code | `local_code` | — | `faa` | 2 |
| operator | — | — | `operator` | 1 |
| owner | — | — | `owner` | 1 |
| runways (list) | `runways.csv` (one-to-many, via `airport_ident`) | — | — | 1 |

**Requirement 4** (≥8 attributes in joint dataset): 12 conceptual attributes above, comfortably over 8.

**Requirement 5** (≥5 attributes in ≥2 datasets, some non-name in all 3): `iata`, `icao`, `coordinates`, `country`, `city` are each in all 3 sources — that's 5 non-name attributes already in all three, not just two.

## Requirements 2 & 3 — entity counts and overlap

Using `icao_code`/`ICAO`/`icao` as the matching key here (normalized: uppercased, whitespace-stripped) — **but only for this feasibility check.** The actual identity resolution deliverable should *not* rely on ICAO/IATA as the matching mechanism: if a shared, mostly-complete identifier drives the matching, there's nothing left to "resolve," which defeats the point of the exercise (and the README's requirement 4 explicitly frames identification as an *attribute combination*, e.g. name + something — not a single ID field). The real matcher will use name + coordinate + country/location similarity across all entities, deliberately not using ICAO/IATA as input.

What ICAO/IATA *are* good for instead: a **ground-truth / evaluation set**. Once the similarity-based matcher produces candidate pairs, we can check them against airports where ICAO agrees exactly to estimate precision/recall — a reliable identifier used to evaluate a matcher, not to build it.

The numbers below are therefore only a lower-bound sanity check that requirement 3 (≥1,000 entities in ≥2 datasets) is achievable at all — the real methodology will find at least this many, likely more once name/coordinate matching also catches pairs with missing or inconsistent codes.

In [2]:
def normalized_codes(series: pd.Series) -> set:
    codes = series.astype(str).str.strip().str.upper()
    codes = codes.replace({"NAN": None, "": None})
    return set(codes.dropna())

oa_icao = normalized_codes(oa["icao_code"])
kg_icao = normalized_codes(kg["ICAO"])
wd_icao = normalized_codes(wd["icao"])

print(f"distinct ICAO codes -> OurAirports: {len(oa_icao):,} | Kaggle: {len(kg_icao):,} | Wikidata: {len(wd_icao):,}")
print()
print(f"ICAO overlap OurAirports & Kaggle:   {len(oa_icao & kg_icao):,}")
print(f"ICAO overlap OurAirports & Wikidata: {len(oa_icao & wd_icao):,}")
print(f"ICAO overlap Kaggle & Wikidata:      {len(kg_icao & wd_icao):,}")
print(f"ICAO present in all 3 sources:       {len(oa_icao & kg_icao & wd_icao):,}")

in_at_least_2 = (oa_icao & kg_icao) | (oa_icao & wd_icao) | (kg_icao & wd_icao)
union_icao = oa_icao | kg_icao | wd_icao

print()
print(f"[Requirement 3] ICAO codes appearing in >=2 datasets: {len(in_at_least_2):,}  (need >= 1,000)")
print(f"[Requirement 2] Union of distinct ICAO codes across all 3: {len(union_icao):,}  (need >= 2,500, good: 10,000-100,000)")

distinct ICAO codes -> OurAirports: 10,530 | Kaggle: 5,089 | Wikidata: 11,522

ICAO overlap OurAirports & Kaggle:   4,700
ICAO overlap OurAirports & Wikidata: 7,786
ICAO overlap Kaggle & Wikidata:      4,540
ICAO present in all 3 sources:       4,365

[Requirement 3] ICAO codes appearing in >=2 datasets: 8,296  (need >= 1,000)
[Requirement 2] Union of distinct ICAO codes across all 3: 14,480  (need >= 2,500, good: 10,000-100,000)


## Requirement 6 — list attribute (ideally)

`data/raw/ourairports/runways.csv` — one row per runway, foreign-keyed to the airport via `airport_ident` (matches `ident` in `airports-ourairports.csv`). Natural list attribute ("runways of an airport"), same shape as "actors of a movie".

In [3]:
runways = pd.read_csv(project_root / "data" / "raw" / "ourairports" / "runways.csv")
runway_counts = runways.groupby("airport_ident").size()

print(f"runway rows: {len(runways):,}")
print(f"distinct airports with >=1 runway: {len(runway_counts):,}")
print(f"airports with >1 runway: {(runway_counts > 1).sum():,} ({(runway_counts > 1).mean():.1%})")
print(f"max runways for one airport: {runway_counts.max()}")
runway_counts.value_counts().sort_index().rename("num_airports")

runway rows: 48,278
distinct airports with >=1 runway: 41,163
airports with >1 runway: 5,481 (13.3%)
max runways for one airport: 11


1     35682
2      4298
3       892
4       199
5        53
6        23
7         9
8         4
9         1
10        1
11        1
Name: num_airports, dtype: int64

## Summary

| # | Requirement | Target | Estimate | Status |
|---|---|---|---|---|
| 1 | 3 different datasets | 3 | 3 (OurAirports, Kaggle, Wikidata) | done |
| 2 | Entities in joint dataset | >=2,500 (good: 10k-100k) | ~14,480 (union of ICAO codes alone) | done |
| 3 | Entities in >=2 datasets | >=1,000 | ~8,296 (ICAO overlap) | done |
| 4 | Attributes in joint dataset | >=8 | 12 mapped | done |
| 5 | Attributes in >=2 datasets (some in 3) | >=5 (5 in all 3) | iata/icao/coordinates/country/city all in 3 | done |
| 6 | List attribute | ideally 1 | runways (`runways.csv`, 41,163 airports, 13.3% with >1 runway) | done |

All requirements, including the "ideally" list attribute, are comfortably met from a feasibility standpoint using coarse ICAO-based matching alone. Next: build proper profiling/extraction/EDA notebooks for OurAirports and Kaggle (mirroring the Wikidata ones), incorporate `runways.csv` into the OurAirports pipeline, then move to real schema mapping + identity resolution (this notebook's ICAO-only matching is a rough estimate, not the final method).